### Problem Statement 3A (ECG)

3A (ECG). Implement ECG anomaly detection for the given ECG dataset using Autoencoder and build the model by using the following steps:
• Dataset: Each row is one ECG signal of 140 time-steps; the last column is the label (1 = Normal, 0 = Abnormal).
• Goal: Identify abnormal ECG signals by training an autoencoder only on normal signals.

Steps:
• Preprocess the data by scaling features.
• Train an autoencoder on normal ECG signals to learn typical heartbeat patterns.
• Calculate reconstruction errors for signals and set a threshold to detect anomalies.
• Compile the models with Optimizer, Loss, and Evaluation Metric

In [ ]:
# Step 1: Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [ ]:
# Step 2: Load the ECG dataset
data = pd.read_csv("../LP IV Updated Datasets/AutoEncoder/ECG/ecg.csv", header=None)

print("Dataset shape:", data.shape)
print("Normal (1) vs Abnormal (0) counts:")
print(data.iloc[:, -1].value_counts())

# Separate features (140 ECG time-steps) and labels (last column)
X = data.iloc[:, :-1].values
y = data.iloc[:, -1].values

# Split into train and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Keep only normal ECG signals (label == 1) for training the autoencoder
X_train_normal = X_train[y_train == 1]

# Scale features using StandardScaler
scaler = StandardScaler()
X_train_normal = scaler.fit_transform(X_train_normal)
X_test = scaler.transform(X_test)

print("Normal training samples:", len(X_train_normal))
print("Test samples:", len(X_test))

In [ ]:
# Step 3: Build Autoencoder using Sequential model
input_dim = X_train_normal.shape[1]  # 140 features

model = Sequential()

# Encoder: compresses the 140 input features down to 8
model.add(Dense(64, activation='relu', input_shape=(input_dim,)))
model.add(Dropout(0.1))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(8, activation='relu'))

# Decoder: reconstructs back from 8 to 140 features
model.add(Dense(16, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.1))
model.add(Dense(input_dim, activation='linear'))

# Compile model
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

In [ ]:
# Step 4: Train autoencoder only on normal ECG signals
history = model.fit(
    X_train_normal, X_train_normal,
    epochs=30,
    batch_size=64,
    validation_data=(X_test, X_test)
)

In [ ]:
# Step 5: Plot training and validation loss
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss (MSE)')
plt.title('Autoencoder Training Loss')
plt.legend()
plt.show()

In [ ]:
# Step 6: Calculate reconstruction errors on normal training signals
train_predictions = model.predict(X_train_normal, verbose=0)
train_errors = np.mean((X_train_normal - train_predictions) ** 2, axis=1)

# Set threshold at 95th percentile of normal training errors
threshold = np.percentile(train_errors, 95)
print("Anomaly Detection Threshold:", threshold)

In [ ]:
# Step 7: Evaluate on test set
test_predictions = model.predict(X_test, verbose=0)
test_errors = np.mean((X_test - test_predictions) ** 2, axis=1)

# If error > threshold, classify as Abnormal (0), else Normal (1)
y_pred = np.where(test_errors > threshold, 0, 1)

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Abnormal', 'Normal']))